# 04 · Dicts and sets

**How this notebook works**
- Run the setup cell below first. Then go top to bottom, running each cell with **Shift+Enter**.
- Gray **example** cells: run them, then change something and run again.
- **Try it** cells: your scratch space for each section.
- **Predict the output**: write your guess in the comment *before* running.
- **Exercises** at the bottom: write your code, run the cell, then run the `check(...)` cell under it.

In [ ]:
# ▶ Run this cell first (Shift+Enter). It loads the test helper and the same imports LeetCode gives you.
import sys, pathlib
sys.path.insert(0, str(next(p for p in (pathlib.Path.cwd() / "_primer", pathlib.Path.cwd() / "python-primer" / "_primer") if p.exists())))
from check import check, summary, lesson
lesson("ex04_dicts_sets")

import bisect, heapq, math, random
from collections import Counter, OrderedDict, defaultdict, deque
from functools import cache, lru_cache
from itertools import accumulate, combinations, groupby, pairwise, permutations, product
from typing import List, Optional

**Warm-up (5 min):** before reading, close everything and write down every trap you remember from lessons 01–03. Then check them against your *My traps* notes in those notebooks.

`dict` is `HashMap`, `set` is `HashSet`. Lookups, inserts and `in` are O(1) on average. This is the single most-used tool in LeetCode: "have I seen this before?" and "how many times?"

## dict

In [ ]:
d = {}                     # or {"a": 1, "b": 2}
d["a"] = 1                 # insert / overwrite
# d["z"]                   # KeyError if missing (uncomment to see)
print(d.get("z"))          # None if missing
print(d.get("z", 0))       # default if missing
print("a" in d)            # key membership, O(1)
d["b"] = 2
del d["a"]                 # remove (KeyError if missing)
print(d.pop("a", None))    # remove, no error if missing
for k in d:                # keys
    print(k)
for k, v in d.items():     # pairs
    print(k, v)
print(list(d.values()))

Dicts keep insertion order.

Counting by hand:

In [ ]:
nums = [3, 1, 3, 2, 3]
count = {}
for x in nums:
    count[x] = count.get(x, 0) + 1
print(count)

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Counter and defaultdict

In [ ]:
from collections import Counter, defaultdict

c = Counter("banana")
print(c)                           # Counter({'a': 3, 'n': 2, 'b': 1})
print(c["z"])                      # 0, missing keys read as 0 (and are NOT inserted)
print(c.most_common(2))            # [('a', 3), ('n', 2)]
print(Counter("listen") == Counter("silent"))   # anagram check in one line
print(Counter("aab") - Counter("abbb"))         # subtract, drops counts <= 0

groups = defaultdict(list)         # missing key -> new empty list
for word in ["eat", "tea", "tan"]:
    groups["".join(sorted(word))].append(word)
print(dict(groups))
seen = defaultdict(int)            # missing key -> 0
seen["x"] += 1
print(dict(seen))

Trap: with `defaultdict`, even *reading* a missing key (`groups[k]`) inserts it. Use `k in groups` to test.

On LeetCode, `collections`, `heapq`, `bisect`, `math`, `itertools` and `functools` are already imported. In an interview, write the import anyway.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Keys must be hashable

Ints, strings, tuples of hashables: fine. Lists, sets, dicts: not allowed as keys or set members.

In [ ]:
d, visited = {}, set()
word = "eat"
d[tuple(sorted(word))] = "group"    # list -> tuple to use as a key
r, c = 2, 3
visited.add((r, c))                 # grid cells as tuples
print(d, visited)
try:
    visited.add([r, c])
except TypeError as e:
    print("TypeError:", e)

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## set

In [ ]:
s = set()                  # NOT {}: that's an empty dict
s = {1, 2, 3}
s.add(4)
# s.remove(9)              # KeyError if missing (uncomment to see)
s.discard(9)               # no error
print(3 in s)              # O(1)
a, b = {1, 2, 3}, {2, 3, 4}
print(a | b, a & b, a - b)      # union, intersection, difference
nums = [1, 2, 2]
print(len(set(nums)) != len(nums))   # has duplicates

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## The two core moves

**Seen-set:** walk once, check before adding.

In [ ]:
def has_duplicate_demo(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False

print(has_duplicate_demo([1, 2, 3, 1]), has_duplicate_demo([1, 2, 3]))

**Value → index map:** Two Sum. Check for the complement *before* inserting the current value, so an element never pairs with itself.

In [ ]:
def two_sum_demo(nums, target):
    index = {}
    for i, x in enumerate(nums):
        if target - x in index:
            return [index[target - x], i]
        index[x] = i

print(two_sum_demo([2, 7, 11, 15], 9))

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Don't mutate while iterating

Adding or removing keys while looping over a dict or set raises `RuntimeError`. Loop over `list(d)` if you need to delete.

In [ ]:
# Try it: experiment with this section here, then run (Shift+Enter)

## Predict the output

For each one: type your guess into the comment line, **then** run the cell. The output is the answer. If you were wrong, figure out why before moving on.

**1.** What does this print?

In [ ]:
# My guess: 
d = {}
d["a"] = d.get("a", 0) + 1
d["a"] = d.get("a", 0) + 1
print(d)

**2.** What does this print?

In [ ]:
# My guess: 
print(type({}), type({1}))

**3.** What does this print?

In [ ]:
# My guess: 
from collections import defaultdict
g = defaultdict(list)
if g["x"]:
    pass
print(len(g))

**4.** What does this print?

In [ ]:
# My guess: 
print(Counter("aab") - Counter("abbb"))

**5.** What does this print?

In [ ]:
# My guess: 
print({[1, 2]: "a"})

<details><summary>Why (open after running all of them)</summary>

1. `{'a': 2}`
2. `<class 'dict'> <class 'set'>`
3. `1` (the read inserted `"x"`)
4. `Counter({'a': 1})` (b would be −2, so it's dropped)
5. `TypeError: unhashable type: 'list'`
</details>

---
# Exercises

Write your code in each cell (replace `raise NotImplementedError`), run it, then run the `check(...)` cell below it. `ok` means done; `FAIL` shows what was expected; `TODO` means not started.

### 1. `two_sum` · LeetCode 1

Indices [i, j] with i < j and nums[i] + nums[j] == target. One pass, O(n).

```
two_sum([2, 7, 11, 15], 9)  ->  [0, 1]
two_sum([3, 2, 4], 6)       ->  [1, 2]
two_sum([3, 3], 6)          ->  [0, 1]
```

In [ ]:
def two_sum(nums: list, target: int) -> list:
    raise NotImplementedError

In [ ]:
check("two_sum")

### 2. `contains_duplicate` · LeetCode 217

Solve it, with a set.

```
contains_duplicate([1, 2, 3, 1])  ->  True
contains_duplicate([1, 2, 3])     ->  False
contains_duplicate([])            ->  False
```

In [ ]:
def contains_duplicate(nums: list) -> bool:
    raise NotImplementedError

In [ ]:
check("contains_duplicate")

### 3. `group_anagrams` · LeetCode 49

Group words that are anagrams. Use a defaultdict(list) keyed by something hashable. Return groups in first-seen order, words in input order.

```
group_anagrams([''])  ->  [['']]
```

In [ ]:
def group_anagrams(words: list) -> list:
    raise NotImplementedError

In [ ]:
check("group_anagrams")

### 4. `first_unique_char` · LeetCode 387

Index of the first character that appears exactly once, else -1.

```
first_unique_char('leetcode')      ->  0
first_unique_char('loveleetcode')  ->  2
first_unique_char('aabb')          ->  -1
```

In [ ]:
def first_unique_char(s: str) -> int:
    raise NotImplementedError

In [ ]:
check("first_unique_char")

### 5. `top_k_frequent` · LeetCode 347

The k most frequent values, most frequent first (no ties in tests).

```
top_k_frequent([1, 1, 1, 2, 2, 3], 2)  ->  [1, 2]
top_k_frequent([4], 1)                 ->  [4]
```

In [ ]:
def top_k_frequent(nums: list, k: int) -> list:
    raise NotImplementedError

In [ ]:
check("top_k_frequent")

### 6. `can_construct` · LeetCode 383

Can ransom be built from magazine's letters (each used once)? Use Counter.

```
can_construct('aa', 'aab')  ->  True
can_construct('aa', 'ab')   ->  False
```

In [ ]:
def can_construct(ransom: str, magazine: str) -> bool:
    raise NotImplementedError

In [ ]:
check("can_construct")

### 7. `common_elements`

Distinct values that appear in both lists, sorted ascending. Use set operations.

```
common_elements([4, 1, 2, 2], [2, 4, 4, 9])  ->  [2, 4]
common_elements([1], [2])                    ->  []
```

In [ ]:
def common_elements(a: list, b: list) -> list:
    raise NotImplementedError

In [ ]:
check("common_elements")

---
## Wrap-up

In [ ]:
summary()

## My traps

*Double-click to edit.* Write the 3–5 things from this lesson that surprised you or that you got wrong, in your own words. You'll use these in the next warm-up and the exit test.

1. 
2. 
3. 

**Done?** All `ok` in the summary → commit and sync → tell Claude **"lesson 04 done."**